In [1]:
import pandas as pd
import random
import ast
from nltk.corpus import wordnet
from copy import deepcopy
from collections import defaultdict
import difflib
import nltk

# Download necessary NLTK data
nltk.download("wordnet")
nltk.download("omw-1.4")

df = pd.read_csv("../../data/processed/10k_data_modified.csv")


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/ritisha21089/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/ritisha21089/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


label_map = {
    'QUANTITY': 0,
    'UNIT': 1,
    'NAME': 2,
    'FORM': 3,
    'STATE': 4,
    'DF': 5,
    'SIZE': 6,
    'PREPROCESSING': 7,
    'O': -100  # Outside token or padding
}

In [2]:
from collections import Counter

all_tags = []
for label in df["Labels"]:
    all_tags.extend(eval(label))  

tag_counts = Counter(all_tags)

print(tag_counts)

Counter({'O': 31086, 'NAME': 17179, 'QUANTITY': 12446, 'STATE': 9860, 'UNIT': 8530, 'FORM': 3991, 'SIZE': 1801, 'DF': 1027})


In [3]:
# Convert string labels to actual lists
df['Labels'] = df['Labels'].apply(ast.literal_eval)

In [4]:
from collections import Counter

# Flatten all labels
all_labels = [label for labels in df['Labels'] for label in labels]
label_counts = Counter(all_labels)

print(label_counts)

# Define rare tags (appearing less than 100 times, and not 'O')
rare_tags = {tag for tag, count in label_counts.items() if count < 4000 and tag != 'O'}

print("Rare tags:", rare_tags)

Counter({'O': 31086, 'NAME': 17179, 'QUANTITY': 12446, 'STATE': 9860, 'UNIT': 8530, 'FORM': 3991, 'SIZE': 1801, 'DF': 1027})
Rare tags: {'DF', 'SIZE', 'FORM'}


In [5]:
train_df = df.sample(frac=0.7, random_state=42)
test_df = df.drop(train_df.index)

In [6]:
print(train_df.dtypes)
print(train_df.isna().sum())
print(train_df.sample(3))

Ingredient_Phrases    object
Labels                object
dtype: object
Ingredient_Phrases    0
Labels                0
dtype: int64
                                     Ingredient_Phrases  \
2925  2 small zucchini -LRB- cut lengthwise in half ...   
4819  1 large mango -LRB- peeled and cut into 1/3-in...   
8982        1/3 cup fresh coriander , chopped & divided   

                                                 Labels  
2925  [QUANTITY, SIZE, NAME, O, O, O, O, O, O, O, O,...  
4819  [QUANTITY, SIZE, NAME, O, O, O, O, O, O, FORM, O]  
8982     [QUANTITY, UNIT, DF, NAME, O, STATE, O, STATE]  


In [7]:
import pandas as pd
import ast
from collections import defaultdict
from copy import deepcopy
import random
from nltk.corpus import wordnet

def build_entity_dict(train_df):
    entity_dict = defaultdict(set)
    for _, row in train_df.iterrows():
        phrase = row['Ingredient_Phrases']
        labels = row['Labels']
        
        if not isinstance(phrase, str) or not isinstance(labels, list):
            continue  # skip rows with missing or invalid data

        words = phrase.split()

        for word, label in zip(words, labels):
            if label != 'O':
                entity_dict[label].add(word)
    
    return {k: list(v) for k, v in entity_dict.items()}

In [8]:
entity_dict = build_entity_dict(df)
entity_dict

{'QUANTITY': ['-9',
  '850',
  '4',
  '252',
  '350',
  '450',
  '550',
  '14',
  '1/2-4',
  '1.5',
  '-70',
  '340',
  '6',
  '900',
  '90',
  '-3/4',
  '-20',
  '7',
  '-6',
  '800',
  '-700',
  '1',
  '-400',
  'two',
  'six',
  '-4',
  '2/3',
  '1/8',
  '120',
  'quarter',
  '-5',
  '-36',
  'lbs',
  '70',
  '1000',
  '20',
  '1/3',
  '1/2',
  '-',
  '190',
  '55',
  '185',
  'and',
  '2',
  '-12',
  '230',
  '1.14',
  '-2',
  '36-38',
  '-1/2',
  '-1/3',
  '590',
  '24',
  '80',
  '180',
  '852',
  '400',
  '260',
  '1500',
  '681',
  '19',
  '250',
  '150',
  '-2/3',
  '-16',
  '115',
  '30',
  '240',
  '660',
  'half',
  '210',
  '-1/4',
  '315',
  '32',
  'a',
  '-30',
  '5',
  '225',
  '65',
  '8',
  '375',
  '-150',
  '45',
  '11',
  '25',
  'three',
  '-14',
  '319',
  '6-8',
  '500',
  '3/8',
  '2-3/4',
  '5/8',
  'to',
  '4-6',
  '170',
  'lb',
  '1200',
  '75',
  '10',
  '0.5',
  '48',
  '18',
  '-7',
  '-3',
  '285',
  '680',
  '9',
  '3',
  '435',
  'liters',
  '100',
 

In [9]:
# --- Synonym Replacement ---
def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name().replace("_", " "))
    synonyms.discard(word)
    return list(synonyms)

def synonym_replace_O(words, labels, prob=0.3):
    new_words = words[:]
    for i, (word, label) in enumerate(zip(words, labels)):
        if label == 'O' and random.random() < prob:
            syns = get_synonyms(word)
            if syns:
                new_words[i] = random.choice(syns)
    return new_words

# --- Entity Swap ---
def entity_swap(words, labels, entity_dict, rare_tags, prob=0.5):
    new_words = words[:]
    for i, label in enumerate(labels):
        if label in rare_tags and label in entity_dict and random.random() < prob:
            new_words[i] = random.choice(entity_dict[label])
    return new_words

# --- Positional Shuffle ---
def chunk_entities(words, labels):
    chunks = []
    current_chunk = []
    current_label = labels[0]
    for word, label in zip(words, labels):
        if label == current_label:
            current_chunk.append((word, label))
        else:
            chunks.append((current_chunk, current_label))
            current_chunk = [(word, label)]
            current_label = label
    chunks.append((current_chunk, current_label))
    return chunks

def positional_augment(words, labels, mode='entity', prob=0.7):
    if random.random() > prob:
        return words, labels
    
    chunks = chunk_entities(words, labels)

    if mode == 'entity':
        entity_chunks = [chunk for chunk in chunks if chunk[1] != 'O']
        random.shuffle(entity_chunks)
        new_chunks = []
        for chunk, label in chunks:
            if label == 'O':
                new_chunks.append((chunk, label))
            else:
                new_chunks.append(entity_chunks.pop(0))
    else:
        o_chunks = [chunk for chunk in chunks if chunk[1] == 'O']
        random.shuffle(o_chunks)
        new_chunks = []
        for chunk, label in chunks:
            if label != 'O':
                new_chunks.append((chunk, label))
            else:
                new_chunks.append(o_chunks.pop(0))

    new_words, new_labels = [], []
    for chunk, _ in new_chunks:
        for word, label in chunk:
            new_words.append(word)
            new_labels.append(label)

    return new_words, new_labels


In [10]:
# --- Main Augmentation Function ---
def augment_ner_sample(text, label_list, entity_dict, rare_tags, use_synonyms=True, use_entity_swap=True, use_positional=True):
    words = text.split()
    labels = label_list[:]
    aug_words = deepcopy(words)

    if use_synonyms:
        aug_words = synonym_replace_O(aug_words, labels)

    if use_entity_swap:
        aug_words = entity_swap(aug_words, labels, entity_dict, rare_tags)

    if use_positional:
        aug_words, labels = positional_augment(aug_words, labels, mode='entity')

    return " ".join(aug_words), labels

# Randomly sample 1500 rows from train_df for augmentation
subset = train_df.sample(n=2000, random_state=42).copy()
augmented_subset = []

for idx, row in subset.iterrows():
    aug_text, aug_labels = augment_ner_sample(
        row['Ingredient_Phrases'],
        row['Labels'],
        entity_dict,
        rare_tags,  # pass rare tag list
        use_synonyms=True,
        use_entity_swap=True,
        use_positional=True
    )
    augmented_subset.append((aug_text, aug_labels))

aug_df = pd.DataFrame(augmented_subset, columns=['Ingredient_Phrases', 'Labels'])

# Show a random sample
import random
sample_index = random.randint(0, len(aug_df)-1)
print("Original:", train_df.iloc[sample_index]['Ingredient_Phrases'], train_df.iloc[sample_index]['Labels'])
print("Augmented:", aug_df.iloc[sample_index]['Ingredient_Phrases'], aug_df.iloc[sample_index]['Labels'])

Original: 1 tablespoon evaporated unbleached cane sugar -LRB- or regular sugar -RRB- ['QUANTITY', 'UNIT', 'STATE', 'STATE', 'NAME', 'NAME', 'O', 'O', 'O', 'NAME', 'O']
Augmented: 3 1/2 tablespoons sesame seeds ['QUANTITY', 'QUANTITY', 'UNIT', 'NAME', 'NAME']


In [11]:
aug_df.shape

(2000, 2)

In [12]:
aug_df

,Ingredient_Phrases,Labels
0,"1 head light sprouts cabbage , about 1 . 5 lbs","[QUANTITY, UNIT, FORM, FORM, NAME, O, O, QUANT..."
1,"lb lb 1 or 1 veal chicken cutlet , pounded slight","[UNIT, UNIT, QUANTITY, O, QUANTITY, NAME, O, O..."
2,radishes bolshy shredded operating room white ...,"[NAME, O, STATE, O, O, O]"
3,4 chicken thighs or 8 chicken drumsticks,"[QUANTITY, NAME, NAME, O, O, NAME, O]"
4,4 1/2 lbs cucumbers,"[QUANTITY, QUANTITY, UNIT, NAME]"
...,...,...
1995,g 230 guacamole,"[UNIT, QUANTITY, NAME]"
1996,1 tablespoon sambal oelek Beaver State tablesp...,"[QUANTITY, UNIT, NAME, NAME, O, UNIT, QUANTITY..."
1997,za'atar spice mix or sumac,"[NAME, NAME, NAME, O, O]"
1998,"yellow cup pepper diced , 1/4","[FORM, UNIT, NAME, STATE, O, QUANTITY]"


In [13]:
train_df.shape

(7549, 2)

In [14]:
train_df

,Ingredient_Phrases,Labels
5579,"3 ounces goat cheese , crumbled","[QUANTITY, UNIT, NAME, NAME, O, STATE]"
8350,"2 cups low sodium chicken broth , boiled until...","[QUANTITY, UNIT, STATE, STATE, NAME, NAME, O, ..."
3614,100 g minced beef,"[QUANTITY, UNIT, STATE, NAME]"
4463,1 -2 cup shredded monterey jack cheese or 1 -2...,"[QUANTITY, QUANTITY, UNIT, STATE, NAME, NAME, ..."
8045,"4 oranges , juice of -LRB- you can slip in som...","[QUANTITY, NAME, O, FORM, O, O, O, O, O, O, O,..."
...,...,...
2437,"3 lbs carrots , peeled and cut diagonally into...","[QUANTITY, UNIT, NAME, O, STATE, O, STATE, O, ..."
6056,4 large chicken breasts,"[QUANTITY, SIZE, NAME, NAME]"
9072,4 cups hot cooked basmati rice,"[QUANTITY, UNIT, STATE, STATE, NAME, NAME]"
817,"2 tablespoons taco seasoning , chipotle flavor","[QUANTITY, UNIT, NAME, NAME, O, NAME, NAME]"


In [15]:
# Step 1: Add the original column names to the augmented data
aug_df.columns = ['Ingredient_Phrases', 'Labels']

# Step 2: Concatenate the original clean data and augmented data
aug_train_df = pd.concat([train_df, aug_df], ignore_index=True)

# Step 3: Verify the merge
print("Final train_df shape:", aug_train_df.shape)

Final train_df shape: (9549, 2)


In [16]:
aug_train_df

,Ingredient_Phrases,Labels
0,"3 ounces goat cheese , crumbled","[QUANTITY, UNIT, NAME, NAME, O, STATE]"
1,"2 cups low sodium chicken broth , boiled until...","[QUANTITY, UNIT, STATE, STATE, NAME, NAME, O, ..."
2,100 g minced beef,"[QUANTITY, UNIT, STATE, NAME]"
3,1 -2 cup shredded monterey jack cheese or 1 -2...,"[QUANTITY, QUANTITY, UNIT, STATE, NAME, NAME, ..."
4,"4 oranges , juice of -LRB- you can slip in som...","[QUANTITY, NAME, O, FORM, O, O, O, O, O, O, O,..."
...,...,...
9544,g 230 guacamole,"[UNIT, QUANTITY, NAME]"
9545,1 tablespoon sambal oelek Beaver State tablesp...,"[QUANTITY, UNIT, NAME, NAME, O, UNIT, QUANTITY..."
9546,za'atar spice mix or sumac,"[NAME, NAME, NAME, O, O]"
9547,"yellow cup pepper diced , 1/4","[FORM, UNIT, NAME, STATE, O, QUANTITY]"


In [ ]:
test_df.shape

(3235, 2)

In [ ]:
test_df

,Ingredient_Phrases,Labels
0,"1 teaspoon wheat germ , if desired -LRB- optio...","['QUANTITY', 'UNIT', 'NAME', 'NAME', 'O', 'O',..."
1,1 batch jerk sauce -LRB- see recipe above -RRB-,"['QUANTITY', 'UNIT', 'NAME', 'NAME', 'O', 'O',..."
2,3 tablespoons kirsch liqueur -LRB- or dry verm...,"['QUANTITY', 'UNIT', 'NAME', 'NAME', 'O', 'O',..."
3,"2 cups vegetable juice , such as V8 brand","['QUANTITY', 'UNIT', 'NAME', 'FORM', 'O', 'O',..."
4,1 tablespoon vegetable oil -LRB- or ghee if yo...,"['QUANTITY', 'UNIT', 'NAME', 'NAME', 'O', 'O',..."
...,...,...
3230,"10 1/2 ounces brie cheese , cut into 1/2 in ch...","['QUANTITY', 'QUANTITY', 'UNIT', 'NAME', 'NAME..."
3231,"1/4 - 1/2 cup queso fresco , cut into 1/2-inch...","['QUANTITY', 'QUANTITY', 'QUANTITY', 'UNIT', '..."
3232,couple drops lemon juice,"['O', 'O', 'NAME', 'FORM']"
3233,-inch pieces green beans,"['O', 'FORM', 'FORM', 'NAME']"


In [ ]:
test_df.to_csv('../../data/augmented/aug_test_df.csv', index=False)
aug_train_df.to_csv('../../data/augmented/aug_train_df.csv', index=False)

: 